In [ ]:
import traceback, json as _json
_result = {"status": "ok", "log": []}
def _log(*a):
    _result["log"].append(" ".join(str(x) for x in a))
def _save():
    import notebookutils
    notebookutils.fs.put("Files/evaluation_result.json", _json.dumps(_result, indent=1), True)

try:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "-q",
                    "fabric-data-agent-sdk"], check=True)

    import pandas as pd
    from fabric.dataagent.evaluation import evaluate_data_agent

    # Ground truth validated against the Lakehouse (03_validation_sql)
    df = pd.DataFrame({
        "question": [
            "How did electronics revenue change between September 2025 and September 2026?",
            "How many stores show low inventory in the inventory snapshot, and which are they?",
            "Which stores still had electronics campaign spend in September 2026?",
            "Which store had the largest percentage decline in electronics revenue year over year?",
            "How many rows are in the new_evidence table right now?",
        ],
        "expected_answer": [
            "Electronics revenue declined from 981,000 in September 2025 to 585,000 "
            "in September 2026, a decrease of about 40.4 percent (40.37%).",
            "Four stores show low inventory: S01, S02, S03 and S04.",
            "Only S01, S02 and S03 kept campaign spend (1,500 each); S04 through S10 "
            "had zero electronics campaign spend in September 2026.",
            "S02 had the largest percentage decline at -54.5 percent.",
            "The new_evidence table is currently empty: 0 rows.",
        ],
    })

    eval_id = evaluate_data_agent(df, data_agent_name="RetailEvidence",
                                  table_name="demo_eval")
    _result["evaluation_id"] = str(eval_id)
    _log("evaluation complete:", eval_id)
    _save()

    rows = spark.sql(
        "SELECT question, expected_answer, actual_answer, evaluation_judgement "
        "FROM demo_eval WHERE evaluation_id = '" + str(eval_id) + "'"
    ).collect()
    _result["graded"] = [
        {"q": r["question"], "judgement": str(r["evaluation_judgement"]),
         "actual": (r["actual_answer"] or "")[:600]}
        for r in rows
    ]
    _log("graded rows:", len(rows))
except Exception:
    _result["status"] = "error"
    _result["log"].append(traceback.format_exc())

_save()
print(_result["status"])
